## Imports y detección de la raíz del proyecto

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("No pude encontrar la raíz del proyecto.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
WORKING_DIR = DATA_DIR / "working"
MANIFESTS_DIR = DATA_DIR / "manifests"
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("WORKING_DIR:", WORKING_DIR)
print("MANIFESTS_DIR:", MANIFESTS_DIR)

PROJECT_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2
DATA_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data
WORKING_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working
MANIFESTS_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests


# rutas base de NIH

In [2]:
meta_dir = WORKING_DIR / "nih" / "meta"

metadata_csv = meta_dir / "Data_Entry_2017.csv"
train_val_file = meta_dir / "train_val_list.txt"
test_file = meta_dir / "test_list.txt"

print("metadata_csv:", metadata_csv)
print("train_val_file:", train_val_file)
print("test_file:", test_file)

print("\n¿Existen?")
print("Data_Entry_2017.csv ->", metadata_csv.exists())
print("train_val_list.txt ->", train_val_file.exists())
print("test_list.txt ->", test_file.exists())

metadata_csv: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/meta/Data_Entry_2017.csv
train_val_file: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/meta/train_val_list.txt
test_file: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/meta/test_list.txt

¿Existen?
Data_Entry_2017.csv -> True
train_val_list.txt -> True
test_list.txt -> True


# cargar metadata original

In [3]:
df = pd.read_csv(metadata_csv)

print("Shape original:", df.shape)
print("\nColumnas originales:")
print(df.columns.tolist())

df.head()

Shape original: (112120, 12)

Columnas originales:
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


# renombrar columnas y agregar columnas base del proyecto

In [4]:
df = df.rename(columns={
    "Image Index": "image_name",
    "Finding Labels": "finding_labels",
    "Patient ID": "patient_id",
    "Patient Age": "patient_age",
    "Patient Gender": "patient_gender",
    "View Position": "view_position",
})

df["dataset_name"] = "NIH_ChestXray14"
df["modality"] = "xray"
df["dimension"] = "2D"

df.head()

,image_name,finding_labels,Follow-up #,patient_id,patient_age,patient_gender,view_position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,dataset_name,modality,dimension
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN,NIH_ChestXray14,xray,2D
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN,NIH_ChestXray14,xray,2D
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN,NIH_ChestXray14,xray,2D
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN,NIH_ChestXray14,xray,2D
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN,NIH_ChestXray14,xray,2D


# parsear etiquetas multilabel

In [5]:
def parse_labels(x):
    if pd.isna(x):
        return []
    return [label.strip() for label in str(x).split("|") if label.strip()]

df["labels_list"] = df["finding_labels"].apply(parse_labels)

df[["image_name", "finding_labels", "labels_list"]].head()

,image_name,finding_labels,labels_list
0,00000001_000.png,Cardiomegaly,[Cardiomegaly]
1,00000001_001.png,Cardiomegaly|Emphysema,"[Cardiomegaly, Emphysema]"
2,00000001_002.png,Cardiomegaly|Effusion,"[Cardiomegaly, Effusion]"
3,00000002_000.png,No Finding,[No Finding]
4,00000003_000.png,Hernia,[Hernia]


# cargar los splits originales

In [6]:
train_val_names = set(pd.read_csv(train_val_file, header=None)[0].astype(str))
test_names = set(pd.read_csv(test_file, header=None)[0].astype(str))

print("Cantidad train_val:", len(train_val_names))
print("Cantidad test:", len(test_names))

Cantidad train_val: 86524
Cantidad test: 25596


# asignar split original

In [7]:
df["split"] = "unknown"
df.loc[df["image_name"].isin(train_val_names), "split"] = "trainval"
df.loc[df["image_name"].isin(test_names), "split"] = "test"

print("Conteo por split original:")
print(df["split"].value_counts(dropna=False))

Conteo por split original:
split
trainval    86524
test        25596
Name: count, dtype: int64


# guardar manifiesto maestro

In [8]:
manifest_master_path = MANIFESTS_DIR / "manifest_nih_master.csv"
df.to_csv(manifest_master_path, index=False)

print("Manifest maestro guardado en:", manifest_master_path)
print("Shape final:", df.shape)

Manifest maestro guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/manifest_nih_master.csv
Shape final: (112120, 17)


# crear split final train/val/test por paciente

In [9]:
RANDOM_SEED = 42
VAL_RATIO = 0.15

df_master = pd.read_csv(manifest_master_path)

trainval_df = df_master[df_master["split"] == "trainval"].copy()
test_df = df_master[df_master["split"] == "test"].copy()

unique_patients = trainval_df["patient_id"].dropna().unique()

rng = np.random.default_rng(RANDOM_SEED)
shuffled_patients = rng.permutation(unique_patients)

n_val_patients = int(len(shuffled_patients) * VAL_RATIO)
val_patients = set(shuffled_patients[:n_val_patients])
train_patients = set(shuffled_patients[n_val_patients:])

train_df = trainval_df[trainval_df["patient_id"].isin(train_patients)].copy()
val_df = trainval_df[trainval_df["patient_id"].isin(val_patients)].copy()

train_df["split_final"] = "train"
val_df["split_final"] = "val"
test_df["split_final"] = "test"

nih_final = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("Conteo final por split:")
print(nih_final["split_final"].value_counts())

print("\nPacientes únicos por split:")
print("train:", train_df["patient_id"].nunique())
print("val:", val_df["patient_id"].nunique())
print("test:", test_df["patient_id"].nunique())

Conteo final por split:
split_final
train    73499
test     25596
val      13025
Name: count, dtype: int64

Pacientes únicos por split:
train: 23807
val: 4201
test: 2797


# guardar manifiesto final

In [10]:
manifest_final_path = MANIFESTS_DIR / "manifest_nih_final.csv"
nih_final.to_csv(manifest_final_path, index=False)

print("Manifest final guardado en:", manifest_final_path)
print("Shape final:", nih_final.shape)

Manifest final guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/manifest_nih_final.csv
Shape final: (112120, 18)


# crear subset pequeño piloto

In [11]:
SUBSET_TRAIN = 500
SUBSET_VAL = 200
SUBSET_TEST = 200
RANDOM_SEED = 42

train_small = (
    nih_final[nih_final["split_final"] == "train"]
    .sample(
        n=min(SUBSET_TRAIN, len(nih_final[nih_final["split_final"] == "train"])),
        random_state=RANDOM_SEED
    )
    .copy()
)

val_small = (
    nih_final[nih_final["split_final"] == "val"]
    .sample(
        n=min(SUBSET_VAL, len(nih_final[nih_final["split_final"] == "val"])),
        random_state=RANDOM_SEED
    )
    .copy()
)

test_small = (
    nih_final[nih_final["split_final"] == "test"]
    .sample(
        n=min(SUBSET_TEST, len(nih_final[nih_final["split_final"] == "test"])),
        random_state=RANDOM_SEED
    )
    .copy()
)

nih_subset_small = pd.concat([train_small, val_small, test_small], ignore_index=True)

print("Conteo subset pequeño:")
print(nih_subset_small["split_final"].value_counts())
print("\nShape subset pequeño:", nih_subset_small.shape)

Conteo subset pequeño:
split_final
train    500
val      200
test     200
Name: count, dtype: int64

Shape subset pequeño: (900, 18)


# guardar subset pequeño

In [12]:
subset_dir = WORKING_DIR / "nih" / "subsets"
subset_dir.mkdir(parents=True, exist_ok=True)

nih_subset_small_path = subset_dir / "nih_subset_small.csv"
nih_subset_small.to_csv(nih_subset_small_path, index=False)

print("Subset pequeño guardado en:", nih_subset_small_path)

Subset pequeño guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/subsets/nih_subset_small.csv


# validación final de archivos generados

In [13]:
print("Archivos generados:")
print("-", manifest_master_path)
print("-", manifest_final_path)
print("-", nih_subset_small_path)

print("\n¿Existen?")
print("manifest_nih_master.csv ->", manifest_master_path.exists())
print("manifest_nih_final.csv ->", manifest_final_path.exists())
print("nih_subset_small.csv ->", nih_subset_small_path.exists())

Archivos generados:
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/manifest_nih_master.csv
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/manifest_nih_final.csv
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/subsets/nih_subset_small.csv

¿Existen?
manifest_nih_master.csv -> True
manifest_nih_final.csv -> True
nih_subset_small.csv -> True


## Nuevo subset mas grande

In [14]:
SUBSET_TRAIN_LARGE = 15000
SUBSET_VAL_LARGE = 2000
SUBSET_TEST_LARGE = 2000
RANDOM_SEED = 42

train_large = (
    nih_final[nih_final["split_final"] == "train"]
    .sample(
        n=min(SUBSET_TRAIN_LARGE, len(nih_final[nih_final["split_final"] == "train"])),
        random_state=RANDOM_SEED
    )
    .copy()
)

val_large = (
    nih_final[nih_final["split_final"] == "val"]
    .sample(
        n=min(SUBSET_VAL_LARGE, len(nih_final[nih_final["split_final"] == "val"])),
        random_state=RANDOM_SEED
    )
    .copy()
)

test_large = (
    nih_final[nih_final["split_final"] == "test"]
    .sample(
        n=min(SUBSET_TEST_LARGE, len(nih_final[nih_final["split_final"] == "test"])),
        random_state=RANDOM_SEED
    )
    .copy()
)

nih_subset_large = pd.concat(
    [train_large, val_large, test_large],
    ignore_index=True
)

print("Conteo subset large:")
print(nih_subset_large["split_final"].value_counts())
print("\nShape subset large:", nih_subset_large.shape)

Conteo subset large:
split_final
train    15000
val       2000
test      2000
Name: count, dtype: int64

Shape subset large: (19000, 18)


In [15]:
subset_dir = DATA_DIR / "working" / "nih" / "subsets"
subset_dir.mkdir(parents=True, exist_ok=True)

nih_subset_large_path = subset_dir / "nih_subset_large.csv"
nih_subset_large.to_csv(nih_subset_large_path, index=False)

print("Subset large guardado en:", nih_subset_large_path)

Subset large guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/subsets/nih_subset_large.csv
